# Backpropagation: Computing Gradients with the Chain Rule

This notebook accompanies the [Backpropagation](../dl-foundations/backpropagation) page.

**Learning objectives:**
- Implement forward + backward pass for a 2-layer MSE network
- Verify gradients with numerical finite differences
- Run a training loop and watch the loss decrease
- Understand the ReLU gradient and the subgradient convention

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('Ready — let\'s implement backprop!')

## Exercise 1: Forward + Backward Pass

Implement the complete forward and backward pass for a 2-layer network with MSE loss.

In [ ]:
import numpy as np
np.random.seed(42)

# Dataset: 4 samples, 3 features → 2 outputs
X = np.random.randn(4, 3)
Y = np.random.randn(4, 2)

# Initialize weights
W1 = np.random.randn(4, 3) * 0.5
b1 = np.zeros(4)
W2 = np.random.randn(2, 4) * 0.5
b2 = np.zeros(2)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1.T + b1
    h1 = np.maximum(0, z1)
    z2 = h1 @ W2.T + b2
    return z1, h1, z2

def backward(X, Y, z1, h1, z2, W2):
    n = X.shape[0]
    delta2 = (2/n) * (z2 - Y)           # (n, 2)
    dW2    = delta2.T @ h1               # (2, 4)
    db2    = delta2.sum(axis=0)           # (2,)
    delta1 = (delta2 @ W2) * (z1 > 0).astype(float)  # (n, 4)
    dW1    = delta1.T @ X                # (4, 3)
    db1    = delta1.sum(axis=0)           # (4,)
    return dW1, db1, dW2, db2

z1, h1, z2 = forward(X, W1, b1, W2, b2)
loss = np.mean((z2 - Y)**2)
dW1, db1, dW2, db2 = backward(X, Y, z1, h1, z2, W2)

print(f'Loss: {loss:.4f}')
print(f'dW1 shape: {dW1.shape}  (should be {W1.shape})')
print(f'dW2 shape: {dW2.shape}  (should be {W2.shape})')

## Exercise 2: Numerical Gradient Check

Verify every gradient using finite differences: `(L(w+ε) - L(w-ε)) / 2ε`

In [ ]:
import numpy as np
np.random.seed(42)

X = np.random.randn(4, 3)
Y = np.random.randn(4, 2)
W1 = np.random.randn(4, 3) * 0.5
b1 = np.zeros(4)
W2 = np.random.randn(2, 4) * 0.5
b2 = np.zeros(2)

def compute_loss(W1, b1, W2, b2):
    z1 = X @ W1.T + b1
    h1 = np.maximum(0, z1)
    z2 = h1 @ W2.T + b2
    return np.mean((z2 - Y)**2)

# Analytical gradients
z1, h1, z2 = forward(X, W1, b1, W2, b2)
dW1, db1, dW2, db2 = backward(X, Y, z1, h1, z2, W2)

# Numerical check for several W1 entries
eps = 1e-5
max_err = 0.0
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        W1p = W1.copy(); W1p[i,j] += eps
        W1m = W1.copy(); W1m[i,j] -= eps
        num_grad = (compute_loss(W1p, b1, W2, b2) - compute_loss(W1m, b1, W2, b2)) / (2*eps)
        err = abs(num_grad - dW1[i,j]) / (abs(num_grad) + 1e-8)
        max_err = max(max_err, err)

print(f'Max relative error across all W1 entries: {max_err:.2e}')
print(f'Gradient check passed: {max_err < 1e-4}  (< 1e-4 means backprop is correct)')

## Exercise 3: Training Loop

Run 200 gradient descent steps and plot the loss curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# Regression dataset
X = np.random.randn(20, 3)
Y = np.random.randn(20, 2)

W1 = np.random.randn(4, 3) * 0.3
b1 = np.zeros(4)
W2 = np.random.randn(2, 4) * 0.3
b2 = np.zeros(2)

lr = 0.01
loss_history = []

for step in range(300):
    # Forward
    z1 = X @ W1.T + b1
    h1 = np.maximum(0, z1)
    z2 = h1 @ W2.T + b2
    loss = np.mean((z2 - Y)**2)
    loss_history.append(loss)

    # Backward
    n = X.shape[0]
    delta2 = (2/n) * (z2 - Y)
    dW2 = delta2.T @ h1
    db2 = delta2.sum(axis=0)
    delta1 = (delta2 @ W2) * (z1 > 0).astype(float)
    dW1 = delta1.T @ X
    db1 = delta1.sum(axis=0)

    # Gradient descent update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

plt.figure(figsize=(8, 5))
plt.plot(loss_history, 'steelblue', linewidth=2)
plt.xlabel('Training step')
plt.ylabel('MSE Loss')
plt.title('Loss Curve: 2-Layer MLP Training')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Initial loss: {loss_history[0]:.4f}')
print(f'Final loss:   {loss_history[-1]:.4f}')

## Next

You have now implemented forward propagation, loss computation, backpropagation, and gradient descent from scratch in NumPy. These are the complete foundations of deep learning.

Continue to [Gradient Descent for Neural Networks](../dl-foundations/gradient-descent-nn) to learn about momentum, Adam, and learning rate schedules.